# PVC-6 Supplementary Final Figures

Clean figure-generation notebook. Loads all results from pickle — no recomputation. Run top-to-bottom to reproduce all supplementary figures.

In [ ]:
import sys, os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/pvc-6/pvc6_helper_modules/')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/shared_helper_modules/')
from pvc6_plotting import *
from pvc6_stim_analysis import *
from spikeparam_plotting import plot_corr_heatmap_only_spk

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sns.set_style('white')
import warnings
warnings.filterwarnings('ignore')
set_plot_style()

PICKLE_DIR = '/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/pvc6_pickles/'
WINDOWS_MS = [5, 25, 50, 100, 200, 300, 400, 500]

def _load(path):
    with open(path, 'rb') as fh:
        return pickle.load(fh)

## Load Pickles

In [ ]:
# ── Cell 1 ────────────────────────────────────────────────────────────────────
df_stim_features_c1, _ = (
    _load(os.path.join(PICKLE_DIR, 'df_stim_features.pkl')),
    _load(os.path.join(PICKLE_DIR, 'all_data.pkl')),
)
logr_accs_c1      = _load(os.path.join(PICKLE_DIR, 'logr_accs.pkl'))
svm_accs_c1       = _load(os.path.join(PICKLE_DIR, 'svm_accs.pkl'))
rf_accs_c1        = _load(os.path.join(PICKLE_DIR, 'rf_accs.pkl'))
rf_model_c1       = _load(os.path.join(PICKLE_DIR, 'rf_model.pkl'))
rf_imps_c1        = _load(os.path.join(PICKLE_DIR, 'rf_importances.pkl'))
results_win_c1    = _load(os.path.join(PICKLE_DIR, 'results_window.pkl'))
df_pink_filt_c1   = _load(os.path.join(PICKLE_DIR, 'df_pink_filtered_c1.pkl'))
print('Cell 1 pickles loaded.')

In [ ]:
# ── Cell 2 ────────────────────────────────────────────────────────────────────
df_stim_features_c2, _ = (
    _load(os.path.join(PICKLE_DIR, 'df_stim_features2.pkl')),
    _load(os.path.join(PICKLE_DIR, 'all_data2.pkl')),
)
logr_accs_c2_bin  = _load(os.path.join(PICKLE_DIR, 'logr_accs2_binary.pkl'))
svm_accs_c2_bin   = _load(os.path.join(PICKLE_DIR, 'svm_accs2_binary.pkl'))
rf_accs_c2_bin    = _load(os.path.join(PICKLE_DIR, 'rf_accs2_binary.pkl'))
rf_model_c2       = _load(os.path.join(PICKLE_DIR, 'rf_model2.pkl'))
rf_imps_c2        = _load(os.path.join(PICKLE_DIR, 'rf_importances2.pkl'))
results_win_c2    = _load(os.path.join(PICKLE_DIR, 'results_window2.pkl'))
df_pink_filt_c2   = _load(os.path.join(PICKLE_DIR, 'df_pink_filtered_c2.pkl'))
print('Cell 2 pickles loaded.')

## Fig S1 — Intra-Spike Feature Correlation Matrix

Lower-triangle Pearson correlation heatmap across all spike waveform features (pink noise spikes only).

In [ ]:
# Cell 1 — pink spikes only (pre-filtered, waveform + stim features)
_drop = ['sweep', 'stim_type', 'pink_type', 'spike_num', 'log_isi']
df_pink_c1 = df_pink_filt_c1.drop(_drop, axis=1, errors='ignore')
spike_features_c1 = [c for c in df_pink_c1.columns
                     if c not in ('stim_exp', 'stim_mean', 'stim_std')]
plot_corr_heatmap_only_spk(df_pink_c1, spike_features_c1, title='Cell 1 — Pink noise spikes')

In [ ]:
# Cell 2 — pink spikes only (pre-filtered, waveform + stim features)
_drop = ['sweep', 'stim_type', 'pink_type', 'spike_num', 'log_isi']
df_pink_c2 = df_pink_filt_c2.drop(_drop, axis=1, errors='ignore')
spike_features_c2 = [c for c in df_pink_c2.columns
                     if c not in ('stim_exp', 'stim_mean', 'stim_std')]
plot_corr_heatmap_only_spk(df_pink_c2, spike_features_c2, title='Cell 2 — Pink noise spikes')

## Fig S2 — Bootstrap Accuracy Distributions

In [ ]:
# Cell 1 — KDE overlay only
plot_bootstrap_histograms([logr_accs_c1, svm_accs_c1, rf_accs_c1],
                          ['Logistic Regression', 'SVM', 'Random Forest'], overlay=True)

In [ ]:
# Cell 2 — binary (constant vs. pink), KDE overlay
plot_bootstrap_histograms([logr_accs_c2_bin, svm_accs_c2_bin, rf_accs_c2_bin],
                          ['Logistic Regression', 'SVM', 'Random Forest'], overlay=True)

## Fig S3 — Window Expansion (R² vs. pre-spike window)

In [ ]:
_shuf_c1 = {f'{w}ms_{t}': results_win_c1[f'shuf_{w}ms_{t}']
            for w in WINDOWS_MS for t in ['stim_mean', 'stim_std', 'stim_exp']
            if f'shuf_{w}ms_{t}' in results_win_c1} or None

_shuf_c2 = {f'{w}ms_{t}': results_win_c2[f'shuf_{w}ms_{t}']
            for w in WINDOWS_MS for t in ['stim_mean', 'stim_std', 'stim_exp']
            if f'shuf_{w}ms_{t}' in results_win_c2} or None

plot_window_expansion(
    results_win_c1, WINDOWS_MS,
    targets=('stim_mean', 'stim_std', 'stim_exp'),
    target_labels=('stim mean', 'stim std', 'stim exp'),
    shuffle_results=_shuf_c1,
    results_by_window_2=results_win_c2,
    shuffle_results_2=_shuf_c2,
    cell_labels=('Cell 1 (SST+)', 'Cell 2 (unmarked)'),
)

## Fig S4 — Beta Weights

In [ ]:
plot_beta_weights_combined(results_win_c1, window_ms=200)

In [ ]:
plot_beta_weights_combined(results_win_c2, window_ms=500)